##Persistența dezechilibrului → preț de dezechilibru

Prețul de dezechilibru depinde doar de dezechilibrul curent sau și de persistența stării SEN?

Folosirea duratei episoadelor consecutive și probabilităților de tranziție calculate de A. Compararea Imbalance Price pentru primul QH de deficit/surplus cu intervalele în care starea persistă 2, 3, 4+ QH. Controlarea pentru magnitudinea dezechilibrului.

In [ ]:
# ============================================================

# TASK 55 — PERSISTENȚA DEZECHILIBRULUI → PREȚ DE DEZECHILIBRU

# REFACTORIZAT STRUCTURAL – FĂRĂ SCHIMBAREA METODOLOGIEI/OUTPUTULUI

# ============================================================


import os

from pathlib import Path


import pandas as pd

import numpy as np

import plotly.graph_objects as go


from google.colab import drive


# ============================================================

# 1. CONFIGURARE

# ============================================================


def configure_task55():


    drive.mount("/content/drive")


    data_dir = Path(

        "/content/drive/MyDrive/Colab Notebooks/Grupa_B/data"

    )


    if not data_dir.is_dir():

        data_dir = Path(

            "/content/drive/MyDrive/Grupa_B/data"

        )


    input_path = (

        data_dir

        / "Task54_Desechilibru_Fizic_Pret_Desechilibru_2025.csv"

    )


    if not input_path.is_file():

        raise FileNotFoundError(

            "❌ Nu a fost găsit fișierul Task 54:\n"

            f"{input_path}\n\n"

            "Rulează mai întâi Task 54 sau verifică "

            "folderul comun Grupa_B/data."

        )


    return data_dir, input_path


# ============================================================

# 2. IDENTIFICARE COLOANE

# ============================================================


def find_column(df, names):


    for name in names:

        if name in df.columns:

            return name


    normalized = {

        str(c).strip().lower(): c

        for c in df.columns

    }


    for name in names:

        if name.strip().lower() in normalized:

            return normalized[name.strip().lower()]


    return None


def identify_task55_columns(df):


    time_col = find_column(

        df,

        [

            "Timestamp",

            "timestamp",

            "DateTime",

            "datetime",

            "Time",

            "time",

            "Interval Start"

        ]

    )


    price_col = find_column(

        df,

        [

            "Imbalance_Price_EUR_MWh",

            "Imbalance Price",

            "Imbalance_Price",

            "Price"

        ]

    )


    imb_col = find_column(

        df,

        [

            "Physical_Imbalance_MWh",

            "Physical Imbalance MWh",

            "Imbalance_MWh",

            "imbalance"

        ]

    )


    state_col = find_column(

        df,

        [

            "State",

            "state",

            "Regime",

            "regime"

        ]

    )


    if any(

        x is None

        for x in [

            time_col,

            price_col,

            imb_col,

            state_col

        ]

    ):

        raise ValueError(

            "❌ Nu au fost identificate toate coloanele necesare."

        )


    return (

        time_col,

        price_col,

        imb_col,

        state_col

    )


# ============================================================

# 3. ÎNCĂRCARE ȘI PREGĂTIRE DATE

# ============================================================


def load_and_prepare_task55(input_path):


    df = pd.read_csv(input_path)


    (

        time_col,

        price_col,

        imb_col,

        state_col

    ) = identify_task55_columns(df)


    # --------------------------------------------------------

    # CORECTIONEAZĂ PROBLEMA NR. 4:

    # Timestamp-ul este definit explicit ca UTC.

    #

    # Timestamp-ul rămâne în UTC pentru calculul continuității

    # și al episoadelor de 15 minute.

    # --------------------------------------------------------


    df[time_col] = pd.to_datetime(

        df[time_col],

        utc=True,

        errors="coerce"

    )


    df[price_col] = pd.to_numeric(

        df[price_col],

        errors="coerce"

    )


    df[imb_col] = pd.to_numeric(

        df[imb_col],

        errors="coerce"

    )


    df = df.dropna(

        subset=[

            time_col,

            price_col,

            imb_col,

            state_col

        ]

    ).copy()


    df = (

        df

        .sort_values(time_col)

        .reset_index(drop=True)

    )


    df["State_Clean"] = (

        df[state_col]

        .astype(str)

        .str.strip()

        .str.lower()

    )


    df = df[

        df["State_Clean"].isin(

            [

                "deficit",

                "surplus"

            ]

        )

    ].copy()


    df["Magnitude_MWh"] = (

        df[imb_col].abs()

    )


    return (

        df,

        time_col,

        price_col,

        imb_col,

        state_col

    )


# ============================================================

# 4. CONSTRUIRE EPISOADE DE PERSISTENȚĂ

# ============================================================


def build_task55_episodes(

    df,

    time_col

):


    time_diff = (

        df[time_col].diff()

    )


    new_episode = (

        df["State_Clean"].ne(

            df["State_Clean"].shift()

        )

        |

        time_diff.gt(

            pd.Timedelta(minutes=15)

        )

    )


    df["Episode_ID"] = (

        new_episode.cumsum()

    )


    df["QH_in_episode"] = (

        df.groupby("Episode_ID")

        .cumcount()

        + 1

    )


    episode_lengths = (

        df.groupby("Episode_ID")

        .size()

        .rename("Episode_Length_QH")

    )


    df = df.join(

        episode_lengths,

        on="Episode_ID"

    )


    episodes = (

        df.groupby("Episode_ID")

        .agg(

            State=(

                "State_Clean",

                "first"

            ),

            Length_QH=(

                "Episode_Length_QH",

                "first"

            )

        )

        .reset_index()

    )


    df["Persistence_Group"] = np.select(

        [

            df["QH_in_episode"] == 1,

            df["QH_in_episode"] == 2,

            df["QH_in_episode"] == 3,

            df["QH_in_episode"] >= 4

        ],

        [

            "QH1",

            "QH2",

            "QH3",

            "QH4+"

        ],

        default="Other"

    )


    first_qh = (

        df[

            df["QH_in_episode"] == 1

        ][

            [

                "Episode_ID",

                "Magnitude_MWh"

            ]

        ]

        .rename(

            columns={

                "Magnitude_MWh":

                    "Initial_Magnitude_MWh"

            }

        )

    )


    df = df.merge(

        first_qh,

        on="Episode_ID",

        how="left"

    )


    return (

        df,

        episodes

    )


# ============================================================

# 5. ZONE MAGNITUDINE

# ============================================================


def add_task55_magnitude_zones(df):


    p25 = (

        df["Initial_Magnitude_MWh"]

        .quantile(0.25)

    )


    p50 = (

        df["Initial_Magnitude_MWh"]

        .quantile(0.50)

    )


    p75 = (

        df["Initial_Magnitude_MWh"]

        .quantile(0.75)

    )


    p90 = (

        df["Initial_Magnitude_MWh"]

        .quantile(0.90)

    )


    p95 = (

        df["Initial_Magnitude_MWh"]

        .quantile(0.95)

    )


    p99 = (

        df["Initial_Magnitude_MWh"]

        .quantile(0.99)

    )


    df["Magnitude_Zone"] = pd.cut(

        df["Initial_Magnitude_MWh"],

        bins=[

            -np.inf,

            p25,

            p50,

            p75,

            p90,

            p95,

            p99,

            np.inf

        ],

        labels=[

            "0-P25",

            "P25-P50",

            "P50-P75",

            "P75-P90",

            "P90-P95",

            "P95-P99",

            "P99+"

        ]

    )


    return df


# ============================================================

# 6. REZUMAT PERSISTENȚĂ → PREȚ

# ============================================================


def build_task55_summary(

    df,

    price_col

):


    return (

        df[

            df["Persistence_Group"].isin(

                [

                    "QH1",

                    "QH2",

                    "QH3",

                    "QH4+"

                ]

            )

        ]

        .groupby(

            [

                "State_Clean",

                "Persistence_Group"

            ],

            observed=True

        )

        .agg(

            N=(

                price_col,

                "size"

            ),

            Median_Price=(

                price_col,

                "median"

            )

        )

        .reset_index()

    )


# ============================================================

# 7. COMPARAȚIE QH1 VS PERSISTENȚĂ

# ============================================================


def build_task55_comparison(

    summary

):


    pivot = summary.pivot(

        index="State_Clean",

        columns="Persistence_Group",

        values="Median_Price"

    )


    comparison_rows = []


    for state in pivot.index:


        qh1 = pivot.loc[state].get(

            "QH1",

            np.nan

        )


        for group in [

            "QH2",

            "QH3",

            "QH4+"

        ]:


            value = pivot.loc[state].get(

                group,

                np.nan

            )


            if (

                pd.notna(value)

                and pd.notna(qh1)

            ):


                comparison_rows.append(

                    {

                        "State": state,

                        "Persistence": group,

                        "QH1_Median_EUR_MWh": qh1,

                        "Persistence_Median_EUR_MWh": value,

                        "Delta_vs_QH1_EUR_MWh":

                            value - qh1

                    }

                )


    return (

        pivot,

        pd.DataFrame(comparison_rows)

    )


# ============================================================

# 8. CONTROL PENTRU MAGNITUDINE

# ============================================================


def build_task55_magnitude_control(

    df,

    price_col

):


    controlled = (

        df[

            df["Persistence_Group"].isin(

                [

                    "QH1",

                    "QH2",

                    "QH3",

                    "QH4+"

                ]

            )

        ]

        .groupby(

            [

                "State_Clean",

                "Magnitude_Zone",

                "Persistence_Group"

            ],

            observed=True

        )

        .agg(

            N=(

                price_col,

                "size"

            ),

            Magnitude_Mean_MWh=(

                "Initial_Magnitude_MWh",

                "mean"

            ),

            Median_Price=(

                price_col,

                "median"

            )

        )

        .reset_index()

    )


    controlled_pivot = (

        controlled.pivot_table(

            index=[

                "State_Clean",

                "Magnitude_Zone"

            ],

            columns="Persistence_Group",

            values="Median_Price",

            observed=True

        )

    )


    controlled_rows = []


    for idx, row in controlled_pivot.iterrows():


        state, zone = idx


        qh1 = row.get(

            "QH1",

            np.nan

        )


        for group in [

            "QH2",

            "QH3",

            "QH4+"

        ]:


            value = row.get(

                group,

                np.nan

            )


            if (

                pd.notna(value)

                and pd.notna(qh1)

            ):


                controlled_rows.append(

                    {

                        "State": state,

                        "Magnitude_Zone": zone,

                        "Persistence": group,

                        "Delta_vs_QH1":

                            value - qh1

                    }

                )


    controlled_comparison = (

        pd.DataFrame(controlled_rows)

    )


    if len(controlled_comparison) > 0:


        control_summary = (

            controlled_comparison

            .groupby(

                [

                    "State",

                    "Persistence"

                ],

                observed=True

            )

            .agg(

                Zones_Comparate=(

                    "Magnitude_Zone",

                    "nunique"

                ),

                Delta_Mediana=(

                    "Delta_vs_QH1",

                    "median"

                )

            )

            .reset_index()

        )


    else:


        control_summary = pd.DataFrame(

            columns=[

                "State",

                "Persistence",

                "Zones_Comparate",

                "Delta_Mediana"

            ]

        )


    return (

        controlled,

        controlled_comparison,

        control_summary

    )


# ============================================================

# 9. PERSISTENȚA EPISOADELOR

# ============================================================


def calculate_task55_persistence(

    episodes

):


    return (

        episodes

        .groupby("State")

        .agg(

            P_QH2_plus=(

                "Length_QH",

                lambda x:

                    (x >= 2).mean() * 100

            ),

            P_QH3_plus=(

                "Length_QH",

                lambda x:

                    (x >= 3).mean() * 100

            ),

            P_QH4_plus=(

                "Length_QH",

                lambda x:

                    (x >= 4).mean() * 100

            )

        )

        .reset_index()

    )


# ============================================================

# 10. REFERINȚĂ GROUP A – TASK 19

# ============================================================


def build_task55_group_a_reference():


    return pd.DataFrame(

        {

            "State": [

                "deficit",

                "surplus"

            ],

            "P_same_15min": [

                81.04,

                82.94

            ],

            "P_same_30min": [

                74.29,

                76.71

            ],

            "P_same_60min": [

                69.75,

                73.28

            ]

        }

    )


# ============================================================

# 11. GRAFIC

# ============================================================


def plot_task55(
    summary,
    out_dir
):
    fig = go.Figure()

    for state in [
        "deficit",
        "surplus"
    ]:
        temp = summary[
            summary["State_Clean"] == state
        ].copy()

        temp["Order"] = (
            temp["Persistence_Group"]
            .map(
                {
                    "QH1": 1,
                    "QH2": 2,
                    "QH3": 3,
                    "QH4+": 4
                }
            )
        )

        temp = temp.sort_values("Order")

        if len(temp) > 0:
            fig.add_trace(
                go.Scatter(
                    x=temp["Persistence_Group"],
                    y=temp["Median_Price"],
                    mode="lines+markers",
                    name=state.capitalize()
                )
            )

    fig.add_hline(
        y=0,
        line_width=0.8
    )

    fig.update_layout(
        title="Task 55 — Prețul dezechilibrului în funcție de persistență",
        xaxis_title="Persistența dezechilibrului",
        yaxis_title="Imbalance Price median [EUR/MWh]",
        template="plotly_white",
        width=900,
        height=500
    )
    fig.update_yaxes(showgrid=True)
    fig.show()

    graph1 = os.path.join(
        out_dir,
        "Task55_Pret_Median_Persistenta.png"
    )

    return graph1


def export_task55(

    summary,

    comparison,

    controlled,

    controlled_comparison,

    persistence,

    groupA_transition,

    out_dir

):


    summary_csv = os.path.join(

        out_dir,

        "Task55_Analiza_Persistenta_Pret_2025.csv"

    )


    comparison_csv = os.path.join(

        out_dir,

        "Task55_QH1_vs_Persistenta_2025.csv"

    )


    controlled_csv = os.path.join(

        out_dir,

        "Task55_Control_Magnitudine_2025.csv"

    )


    controlled_comparison_csv = os.path.join(

        out_dir,

        "Task55_Comparatie_Controlata_2025.csv"

    )


    persistence_csv = os.path.join(

        out_dir,

        "Task55_Persistenta_Episoade_2025.csv"

    )


    groupA_csv = os.path.join(

        out_dir,

        "Task55_Referinta_GroupA_Task19.csv"

    )


    summary.to_csv(

        summary_csv,

        index=False

    )


    comparison.to_csv(

        comparison_csv,

        index=False

    )


    controlled.to_csv(

        controlled_csv,

        index=False

    )


    controlled_comparison.to_csv(

        controlled_comparison_csv,

        index=False

    )


    persistence.to_csv(

        persistence_csv,

        index=False

    )


    groupA_transition.to_csv(

        groupA_csv,

        index=False

    )


    excel_path = os.path.join(

        out_dir,

        "Task55_Rezultate_Complete_2025.xlsx"

    )


    with pd.ExcelWriter(

        excel_path,

        engine="openpyxl"

    ) as writer:


        summary.to_excel(

            writer,

            sheet_name="Persistenta_Pret",

            index=False

        )


        comparison.to_excel(

            writer,

            sheet_name="QH1_vs_Persistenta",

            index=False

        )


        controlled.to_excel(

            writer,

            sheet_name="Control_Magnitudine",

            index=False

        )


        controlled_comparison.to_excel(

            writer,

            sheet_name="Comparatie_Controlata",

            index=False

        )


        persistence.to_excel(

            writer,

            sheet_name="Persistenta_Episoade",

            index=False

        )


        groupA_transition.to_excel(

            writer,

            sheet_name="GroupA_Task19",

            index=False

        )


    return {

        "summary_csv": summary_csv,

        "comparison_csv": comparison_csv,

        "controlled_csv": controlled_csv,

        "controlled_comparison_csv":

            controlled_comparison_csv,

        "persistence_csv": persistence_csv,

        "groupA_csv": groupA_csv,

        "excel_path": excel_path

    }


# ============================================================

# 13. AFIȘARE OUTPUT

# ============================================================


def print_task55_outputs(

    comparison,

    control_summary,

    persistence,

    groupA_transition,

    pivot

):


    print(

        "\n" + "=" * 65

    )


    print(

        "TASK 55 — PERSISTENȚA DEZECHILIBRULUI → PREȚ"

    )


    print(

        "=" * 65

    )


    print(

        "\n1. COMPARAȚIA IMBALANCE PRICE"

    )


    print(

        "-" * 65

    )


    print(

        comparison[

            [

                "State",

                "Persistence",

                "QH1_Median_EUR_MWh",

                "Persistence_Median_EUR_MWh",

                "Delta_vs_QH1_EUR_MWh"

            ]

        ]

        .rename(

            columns={

                "State": "Stare",

                "Persistence": "Persistență",

                "QH1_Median_EUR_MWh":

                    "QH1 median [EUR/MWh]",

                "Persistence_Median_EUR_MWh":

                    "Mediană [EUR/MWh]",

                "Delta_vs_QH1_EUR_MWh":

                    "Δ față de QH1 [EUR/MWh]"

            }

        )

        .to_string(index=False)

    )


    print(

        "\n2. CONTROL PENTRU MAGNITUDINE"

    )


    print(

        "-" * 65

    )


    if len(control_summary) > 0:


        print(

            control_summary

            .rename(

                columns={

                    "State": "Stare",

                    "Persistence": "Persistență",

                    "Zones_Comparate":

                        "Zone magnitudine",

                    "Delta_Mediana":

                        "Δ mediană vs QH1 [EUR/MWh]"

                }

            )

            .to_string(index=False)

        )


    else:


        print(

            "Nu există suficiente observații."

        )


    print(

        "\n3. PERSISTENȚA EPISOADELOR"

    )


    print(

        "-" * 65

    )


    print(

        persistence

        .rename(

            columns={

                "State": "Stare",

                "P_QH2_plus":

                    "≥2 QH [%]",

                "P_QH3_plus":

                    "≥3 QH [%]",

                "P_QH4_plus":

                    "≥4 QH [%]"

            }

        )

        .to_string(index=False)

    )


    print(

        "\n4. GROUP A — TASK 19"

    )


    print(

        "-" * 65

    )


    print(

        groupA_transition

        .rename(

            columns={

                "State": "Stare",

                "P_same_15min":

                    "Aceeași stare 15 min [%]",

                "P_same_30min":

                    "Aceeași stare 30 min [%]",

                "P_same_60min":

                    "Aceeași stare 60 min [%]"

            }

        )

        .to_string(index=False)

    )


    print(

        "\n" + "=" * 65

    )


    print(

        "CONCLUZIE"

    )


    print(

        "=" * 65

    )


    print(

        f"Deficit: mediana scade de la "

        f"{pivot.loc['deficit', 'QH1']:.2f} EUR/MWh "

        f"în QH1 la "

        f"{pivot.loc['deficit', 'QH4+']:.2f} EUR/MWh "

        f"în QH4+."

    )


    print(

        f"Surplus: mediana crește de la "

        f"{pivot.loc['surplus', 'QH1']:.2f} EUR/MWh "

        f"în QH1 la "

        f"{pivot.loc['surplus', 'QH4+']:.2f} EUR/MWh "

        f"în QH4+."

    )


    print(

        "Controlul pe magnitudine este inclus "

        "pentru a verifica relația dintre "

        "persistență și preț la dezechilibre "

        "de magnitudine comparabilă."

    )


    print(

        "Probabilitățile Group A Task 19 sunt "

        "prezentate ca referință pentru "

        "persistența stărilor."

    )


    print(

        "\n" + "=" * 65

    )


    print(

        "✅ TASK 55 FINALIZAT"

    )


    print(

        "=" * 65

    )


# ============================================================

# 14. EXECUȚIE TASK 55

# ============================================================


def run_task55():


    (

        data_dir,

        input_path

    ) = configure_task55()


    (

        df,

        time_col,

        price_col,

        imb_col,

        state_col

    ) = load_and_prepare_task55(

        input_path

    )


    (

        df,

        episodes

    ) = build_task55_episodes(

        df,

        time_col

    )


    df = add_task55_magnitude_zones(

        df

    )


    summary = build_task55_summary(

        df,

        price_col

    )


    (

        pivot,

        comparison

    ) = build_task55_comparison(

        summary

    )


    (

        controlled,

        controlled_comparison,

        control_summary

    ) = build_task55_magnitude_control(

        df,

        price_col

    )


    persistence = calculate_task55_persistence(

        episodes

    )


    groupA_transition = (

        build_task55_group_a_reference()

    )


    plot_task55(

        summary,

        os.path.dirname(input_path)

    )


    export_task55(

        summary,

        comparison,

        controlled,

        controlled_comparison,

        persistence,

        groupA_transition,

        os.path.dirname(input_path)

    )


    print_task55_outputs(

        comparison,

        control_summary,

        persistence,

        groupA_transition,

        pivot

    )


# ============================================================

# 15. RUN

# ============================================================


run_task55()


### Concluzii

- Deficit: mediana scade de la 192.32 EUR/MWh în QH1 la 17.48 EUR/MWh în QH4+.
- Surplus: mediana crește de la 783.16 EUR/MWh în QH1 la 999.11 EUR/MWh în QH4+.



##Regim SEN → distribuția prețurilor

Cum diferă PZU, ID și Imbalance Price între regimurile fundamentale ale SEN?

Folosirea regimurilor Strong Surplus / Surplus / Balanced / Deficit / Strong Deficit și/sau Normal / Stressed / Extreme construite de A. Pentru fiecare se analizează PZU, VWAP ID, ID−PZU, volatilitatea ID și Imbalance Price, inclusiv probabilitățile P90/P95/P99. Respectarea momentului de disponibilitate al variabilelor folosite la definirea regimului.

In [ ]:
# ============================================================

# TASK 62 — REGIM SEN → DISTRIBUȚIA PREȚURILOR

# REFACTORIZAT STRUCTURAL – FĂRĂ SCHIMBAREA METODOLOGIEI/OUTPUTULUI

# OUTPUT: DOAR TABEL + 4 GRAFICE

# ============================================================


import os

from pathlib import Path


import numpy as np

import pandas as pd

import plotly.graph_objects as go


from google.colab import drive

from IPython.display import display


# ============================================================

# 1. CONFIGURARE

# ============================================================


def configure_task62():


    drive.mount("/content/drive")


    data_dir = Path(

        "/content/drive/MyDrive/Colab Notebooks/Grupa_B/data"

    )


    if not data_dir.is_dir():

        data_dir = Path(

            "/content/drive/MyDrive/Grupa_B/data"

        )


    base = str(data_dir)


    paths = {

        "data_dir": data_dir,


        "base": base,


        "task62": (

            data_dir

            / "Task62_PZU_vs_Intraday_2025.csv"

        ),


        "task54": (

            data_dir

            / "Task54_Desechilibru_Fizic_Pret_Desechilibru_2025.csv"

        ),


        "task17": Path(

            "/content/drive/MyDrive/Grupa_B/data/"

            "Task17_Import_Export_vs_Deficit_Surplus_2025_FINAL.csv"

        ),


        "out_csv": (

            data_dir

            / "Task62_Regim_SEN_Distributie_Preturi_2025.csv"

        ),


        "out_xlsx": (

            data_dir

            / "Task62_Regim_SEN_Distributie_Preturi_2025.xlsx"

        ),


        "bnr": (

            data_dir

            / "bnr_eurron_2025.csv"

        )

    }


    for path, name in [

        (

            paths["task62"],

            "Task62_PZU_vs_Intraday_2025.csv"

        ),

        (

            paths["task17"],

            "Task17_Import_Export_vs_Deficit_Surplus_2025_FINAL.csv"

        ),

        (

            paths["task54"],

            "Task54_Desechilibru_Fizic_Pret_Desechilibru_2025.csv"

        )

    ]:


        if not path.is_file():


            raise FileNotFoundError(

                f"❌ Lipsește {name}:\n"

                f"{path}\n\n"

                "Verifică structura folderelor din Google Drive."

            )


    return paths


# ============================================================

# 2. ÎNCĂRCARE SURSE

# ============================================================


def load_task62_sources(paths):


    df62 = pd.read_csv(

        paths["task62"]

    )


    df17 = pd.read_csv(

        paths["task17"]

    )


    df54 = pd.read_csv(

        paths["task54"]

    )


    # --------------------------------------------------------

    # IMPORTANT:

    # Timestamp-urile utilizate pentru alinierea dintre

    # surse sunt păstrate în UTC.

    #

    # Nu convertim aici la Europe/Bucharest deoarece Task 62

    # aliniază serii provenite din surse diferite pe același

    # moment fizic.

    # --------------------------------------------------------


    df62["Hour"] = (

        pd.to_datetime(

            df62["Hour"],

            utc=True,

            errors="coerce"

        )

        .dt.floor("h")

    )


    df17["Timestamp"] = (

        pd.to_datetime(

            df17["Timestamp"],

            utc=True,

            errors="coerce"

        )

        .dt.floor("h")

    )


    df54["Timestamp"] = pd.to_datetime(

        df54["Timestamp"],

        utc=True,

        errors="coerce"

    )


    return (

        df62,

        df17,

        df54

    )


# ============================================================

# 3. PREGĂTIRE REGIM SEN

# ============================================================


def prepare_task62_regime(

    df17

):


    regime_order = [

        "Export puternic",

        "Export moderat",

        "Pozitie neutra",

        "Import moderat",

        "Import puternic"

    ]


    regime = df17[

        [

            "Timestamp",

            "NetImport_DA",

            "Regim_DA"

        ]

    ].copy()


    regime["NetImport_DA"] = pd.to_numeric(

        regime["NetImport_DA"],

        errors="coerce"

    )


    regime["Regim_DA"] = (

        regime["Regim_DA"]

        .astype(str)

        .str.strip()

    )


    regime = (

        regime

        .dropna(

            subset=[

                "Timestamp",

                "NetImport_DA"

            ]

        )

        .drop_duplicates(

            subset=["Timestamp"],

            keep="first"

        )

    )


    regime = regime[

        regime["Regim_DA"].isin(

            regime_order

        )

    ].copy()


    return (

        regime,

        regime_order

    )


# ============================================================

# 4. PREGĂTIRE PIAȚĂ

# ============================================================


def prepare_task62_market(

    df62,

    bnr_path

):


    market = df62[

        [

            "Hour",

            "ID_VWAP_EUR_MWh",

            "PZU_Price_RON_MWh"

        ]

    ].copy()


    market["ID_VWAP_EUR_MWh"] = pd.to_numeric(

        market["ID_VWAP_EUR_MWh"],

        errors="coerce"

    )


    market["PZU_Price_RON_MWh"] = pd.to_numeric(

        market["PZU_Price_RON_MWh"],

        errors="coerce"

    )


    if "ID_Std_QH_EUR_MWh" in df62.columns:


        market["ID_Volatilitate_EUR_MWh"] = (

            pd.to_numeric(

                df62["ID_Std_QH_EUR_MWh"],

                errors="coerce"

            )

        )


    else:


        market["ID_Volatilitate_EUR_MWh"] = (

            pd.to_numeric(

                df62["ID_Range_EUR_MWh"],

                errors="coerce"

            )

        )


    if not bnr_path.is_file():


        raise FileNotFoundError(

            f"❌ Fișierul BNR nu există:\n"

            f"{bnr_path}"

        )


    bnr = pd.read_csv(

        bnr_path

    )


    required_bnr = {

        "date",

        "eur_ron"

    }


    missing_bnr = (

        required_bnr

        -

        set(bnr.columns)

    )


    if missing_bnr:


        raise ValueError(

            "❌ Fișierul BNR nu conține "

            "coloanele necesare: "

            +

            ", ".join(

                sorted(missing_bnr)

            )

        )


    bnr["date"] = pd.to_datetime(

        bnr["date"],

        errors="coerce"

    )


    bnr["eur_ron"] = pd.to_numeric(

        bnr["eur_ron"],

        errors="coerce"

    )


    bnr = (

        bnr

        .dropna(

            subset=[

                "date",

                "eur_ron"

            ]

        )

        .drop_duplicates(

            "date",

            keep="last"

        )

        .sort_values("date")

    )


    # --------------------------------------------------------

    # Pentru asocierea cu cursul BNR se utilizează ziua

    # calendaristică din România.

    #

    # Momentul inițial rămâne UTC; doar aici extragem data

    # locală Europe/Bucharest.

    # --------------------------------------------------------


    market["Date"] = (

        pd.to_datetime(

            market["Hour"],

            utc=True

        )

        .dt.tz_convert(

            "Europe/Bucharest"

        )

        .dt.normalize()

        .dt.tz_localize(None)

    )


    market = pd.merge_asof(

        market.sort_values("Date"),


        bnr.rename(

            columns={

                "date": "Date",

                "eur_ron": "EURRON"

            }

        ),


        on="Date",

        direction="backward"

    )


    market["PZU_EUR_MWh"] = (

        market["PZU_Price_RON_MWh"]

        /

        market["EURRON"]

    )


    market["ID_PZU_Basis_EUR_MWh"] = (

        market["ID_VWAP_EUR_MWh"]

        -

        market["PZU_EUR_MWh"]

    )


    return market


# ============================================================

# 5. PREGĂTIRE DEZECHILIBRU

# ============================================================


def prepare_task62_imbalance(

    df54

):


    imb = df54[

        [

            "Timestamp",

            "Imbalance_Price_EUR_MWh"

        ]

    ].copy()


    imb["Imbalance_Price_EUR_MWh"] = pd.to_numeric(

        imb["Imbalance_Price_EUR_MWh"],

        errors="coerce"

    )


    imb = imb.dropna()


    # --------------------------------------------------------

    # Alinierea orară se face în UTC, la fel ca în celelalte

    # surse. Nu interpretăm ora UTC drept oră locală.

    # --------------------------------------------------------


    imb["Hour"] = (

        imb["Timestamp"]

        .dt.floor("h")

    )


    return (

        imb

        .groupby("Hour")[

            "Imbalance_Price_EUR_MWh"

        ]

        .median()

        .reset_index()

    )


# ============================================================

# 6. ÎMBINARE DATE

# ============================================================


def merge_task62_data(

    market,

    regime,

    regime_order,

    imb_hourly

):


    data = (

        market


        .merge(

            regime[

                [

                    "Timestamp",

                    "NetImport_DA",

                    "Regim_DA"

                ]

            ],


            left_on="Hour",

            right_on="Timestamp",

            how="inner"

        )


        .merge(

            imb_hourly,

            on="Hour",

            how="left"

        )

    )


    data = data[

        data["Regim_DA"].isin(

            regime_order

        )

    ].copy()


    return data


# ============================================================

# 7. FUNCȚIE QUANTILE

# ============================================================


def quantile_value(

    df,

    col,

    p

):


    x = pd.to_numeric(

        df[col],

        errors="coerce"

    ).dropna()


    if x.empty:

        return np.nan


    return x.quantile(p)


# ============================================================

# 8. TABEL FINAL

# ============================================================


def build_task62_summary(

    data,

    regime_order

):


    rows = []


    for regim in regime_order:


        d = data[

            data["Regim_DA"] == regim

        ].copy()


        rows.append(

            {

                "Regim SEN": regim,


                "N": len(d),


                "PZU P50":

                    quantile_value(

                        d,

                        "PZU_EUR_MWh",

                        .50

                    ),


                "PZU P90":

                    quantile_value(

                        d,

                        "PZU_EUR_MWh",

                        .90

                    ),


                "PZU P95":

                    quantile_value(

                        d,

                        "PZU_EUR_MWh",

                        .95

                    ),


                "PZU P99":

                    quantile_value(

                        d,

                        "PZU_EUR_MWh",

                        .99

                    ),


                "VWAP ID P50":

                    quantile_value(

                        d,

                        "ID_VWAP_EUR_MWh",

                        .50

                    ),


                "VWAP ID P90":

                    quantile_value(

                        d,

                        "ID_VWAP_EUR_MWh",

                        .90

                    ),


                "VWAP ID P95":

                    quantile_value(

                        d,

                        "ID_VWAP_EUR_MWh",

                        .95

                    ),


                "VWAP ID P99":

                    quantile_value(

                        d,

                        "ID_VWAP_EUR_MWh",

                        .99

                    ),


                "ID-PZU P50":

                    quantile_value(

                        d,

                        "ID_PZU_Basis_EUR_MWh",

                        .50

                    ),


                "ID-PZU P90":

                    quantile_value(

                        d,

                        "ID_PZU_Basis_EUR_MWh",

                        .90

                    ),


                "ID-PZU P95":

                    quantile_value(

                        d,

                        "ID_PZU_Basis_EUR_MWh",

                        .95

                    ),


                "ID-PZU P99":

                    quantile_value(

                        d,

                        "ID_PZU_Basis_EUR_MWh",

                        .99

                    ),


                "Volatilitate ID P50":

                    quantile_value(

                        d,

                        "ID_Volatilitate_EUR_MWh",

                        .50

                    ),


                "Volatilitate ID P90":

                    quantile_value(

                        d,

                        "ID_Volatilitate_EUR_MWh",

                        .90

                    ),


                "Volatilitate ID P95":

                    quantile_value(

                        d,

                        "ID_Volatilitate_EUR_MWh",

                        .95

                    ),


                "Volatilitate ID P99":

                    quantile_value(

                        d,

                        "ID_Volatilitate_EUR_MWh",

                        .99

                    ),


                "Imbalance Price P50":

                    quantile_value(

                        d,

                        "Imbalance_Price_EUR_MWh",

                        .50

                    ),


                "Imbalance Price P90":

                    quantile_value(

                        d,

                        "Imbalance_Price_EUR_MWh",

                        .90

                    ),


                "Imbalance Price P95":

                    quantile_value(

                        d,

                        "Imbalance_Price_EUR_MWh",

                        .95

                    ),


                "Imbalance Price P99":

                    quantile_value(

                        d,

                        "Imbalance_Price_EUR_MWh",

                        .99

                    )

            }

        )


    return pd.DataFrame(

        rows

    )


# ============================================================

# 9. EXPORT

# ============================================================


def export_task62(

    summary,

    paths

):


    summary.to_csv(

        paths["out_csv"],

        index=False

    )


    summary.to_excel(

        paths["out_xlsx"],

        index=False

    )


# ============================================================

# 10. GRAFIC DISTRIBUȚIE

# ============================================================


def plot_task62_distribution(
    summary,
    regime_order,
    columns,
    labels,
    title,
    output_path,
    zero_line=False
):
    fig = go.Figure()

    for col, label in zip(columns, labels):
        fig.add_trace(
            go.Scatter(
                x=regime_order,
                y=summary[col],
                mode="lines+markers",
                name=label
            )
        )

    if zero_line:
        fig.add_hline(
            y=0,
            line_width=0.8
        )

    fig.update_layout(
        title=title,
        xaxis_title="Regim SEN",
        yaxis_title="EUR/MWh",
        template="plotly_white",
        width=1100,
        height=600
    )
    fig.update_xaxes(tickangle=20)
    fig.update_yaxes(showgrid=True)
    fig.show()


def plot_task62_all(

    summary,

    regime_order,

    base

):


    plot_task62_distribution(

        summary,

        regime_order,


        [

            "PZU P50",

            "PZU P90",

            "PZU P95",

            "PZU P99"

        ],


        [

            "P50",

            "P90",

            "P95",

            "P99"

        ],


        "PZU — distribuția pe regim SEN",


        os.path.join(

            base,

            "Task62_Regim_SEN_PZU.png"

        )

    )


    plot_task62_distribution(

        summary,

        regime_order,


        [

            "VWAP ID P50",

            "VWAP ID P90",

            "VWAP ID P95",

            "VWAP ID P99"

        ],


        [

            "P50",

            "P90",

            "P95",

            "P99"

        ],


        "VWAP Intraday — distribuția pe regim SEN",


        os.path.join(

            base,

            "Task62_Regim_SEN_VWAP_ID.png"

        )

    )


    plot_task62_distribution(

        summary,

        regime_order,


        [

            "ID-PZU P50",

            "ID-PZU P90",

            "ID-PZU P95",

            "ID-PZU P99"

        ],


        [

            "P50",

            "P90",

            "P95",

            "P99"

        ],


        "ID–PZU — distribuția pe regim SEN",


        os.path.join(

            base,

            "Task62_Regim_SEN_ID_PZU.png"

        ),


        zero_line=True

    )


    plot_task62_distribution(

        summary,

        regime_order,


        [

            "Imbalance Price P50",

            "Imbalance Price P90",

            "Imbalance Price P95",

            "Imbalance Price P99"

        ],


        [

            "P50",

            "P90",

            "P95",

            "P99"

        ],


        "Imbalance Price — distribuția pe regim SEN",


        os.path.join(

            base,

            "Task62_Regim_SEN_Imbalance_Price.png"

        )

    )


# ============================================================

# 12. EXECUȚIE TASK 62

# ============================================================


def run_task62():


    paths = configure_task62()


    (

        df62,

        df17,

        df54

    ) = load_task62_sources(

        paths

    )


    (

        regime,

        regime_order

    ) = prepare_task62_regime(

        df17

    )


    market = prepare_task62_market(

        df62,

        paths["bnr"]

    )


    imb_hourly = prepare_task62_imbalance(

        df54

    )


    data = merge_task62_data(

        market,

        regime,

        regime_order,

        imb_hourly

    )


    summary = build_task62_summary(

        data,

        regime_order

    )


    export_task62(

        summary,

        paths

    )


    display(

        summary.round(2)

    )


    plot_task62_all(

        summary,

        regime_order,

        paths["base"]

    )


# ============================================================

# 13. RUN

# ============================================================


run_task62()


### Concluzii

- Regimurile SEN afectează mai puternic valorile extreme decât nivelurile mediane ale prețurilor. Diferențele dintre P50 și cuantilele superioare P90–P99 evidențiază o creștere a dispersiei în regimurile caracterizate de dezechilibre mai pronunțate ale sistemului.

- Volatilitatea Intraday variază în funcție de regimul SEN. Indicatorii de volatilitate calculați pe cuantile arată că modificarea poziției sistemului între export și import este însoțită de modificări ale amplitudinii variațiilor de preț pe piața Intraday.

- Prețul de dezechilibru are o distribuție mai sensibilă la regimul SEN decât prețurile de piață. Diferențele dintre P50 și cuantilele superioare P90–P99 ale Imbalance Price indică apariția unor valori extreme în anumite regimuri, evidențiind impactul poziției SEN asupra costurilor de dezechilibru.

##Dezechilibru SEN → preț de dezechilibru

Care este relația dintre direcția și magnitudinea fizică a dezechilibrului și prețul de dezechilibru?

Compararea System Imbalance MWh cu Imbalance Price, separat pentru Deficit și Surplus. Analiza pe percentile ale magnitudinii și estimarea relațiilor neliniare. Identificarea zonelor în care creșterea dezechilibrului este asociată cu variații disproporționate ale prețului.

In [ ]:
# ================================================================

# TASK 54 – DEZECHILIBRU SEN → PREȚ DE DEZECHILIBRU

# REFACTORIZAT STRUCTURAL – FĂRĂ SCHIMBAREA METODOLOGIEI/OUTPUTULUI

# ================================================================


import os

from pathlib import Path


import numpy as np

import pandas as pd

import plotly.graph_objects as go


from google.colab import drive

from IPython.display import display


def configure_task54():

    drive.mount("/content/drive")


    data_dir = Path("/content/drive/MyDrive/Colab Notebooks/Grupa_B/data")

    if not data_dir.is_dir():

        data_dir = Path("/content/drive/MyDrive/Grupa_B/data")


    paths = {

        "data_dir": data_dir,

        "input_csv": data_dir / "Task54_Desechilibru_Fizic_Pret_Desechilibru_2025.csv",

        "raw_xlsx": data_dir / "Task54_Desechilibru_Fizic_Pret_Desechilibru_2025.xlsx",

        "analysis_csv": data_dir / "Task54_Analiza_Desechilibru_Pret_2025.csv",

        "analysis_xlsx": data_dir / "Task54_Analiza_Desechilibru_Pret_2025.xlsx",

        "graph1": data_dir / "Task54_Grafic1_Pret_Median_Magnitudine.png",

        "graph2": data_dir / "Task54_Grafic2_Desechilibru_vs_Pret.png",

    }


    if not paths["input_csv"].is_file():

        raise FileNotFoundError(

            f"❌ Fișierul Task 54 nu există:\n{paths['input_csv']}\n\nVerifică folderul comun Grupa_B/data."

        )


    return paths


def load_task54_data(input_csv):

    print("=" * 80)

    print("TASK 54 – ANALIZĂ DIN DATELE SALVATE")

    print("=" * 80)


    df = pd.read_csv(input_csv)


    print(f"✅ Fișier citit: {input_csv}")

    print(f"✅ Observații citite: {len(df):,}")

    print()


    df["Timestamp"] = pd.to_datetime(

        df["Timestamp"],

        utc=True,

        errors="coerce"

    )


    numeric_cols = [

        "Imbalance_Price_EUR_MWh",

        "Imbalance_A01_MWh",

        "Imbalance_A02_MWh",

        "Physical_Imbalance_MWh",

        "Physical_Imbalance_Abs_MWh"

    ]


    for col in numeric_cols:

        if col in df.columns:

            df[col] = pd.to_numeric(df[col], errors="coerce")


    required_cols = [

        "Timestamp",

        "Imbalance_Price_EUR_MWh",

        "Imbalance_A01_MWh",

        "Imbalance_A02_MWh",

        "Physical_Imbalance_MWh",

        "Physical_Imbalance_Abs_MWh",

        "State"

    ]


    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:

        raise ValueError(

            "❌ Lipsesc coloanele necesare:\n" + "\n".join(missing_cols)

        )


    if "A86_Available" in df.columns:

        df["A86_Available"] = (

            df["A86_Available"]

            .astype(str)

            .str.lower()

            .map({"true": True, "false": False})

        )


        df_valid = df[df["A86_Available"] == True].copy()

    else:

        df_valid = df.dropna(

            subset=["Imbalance_A01_MWh", "Imbalance_A02_MWh"]

        ).copy()


    df_valid["Physical_Imbalance_MWh"] = (

        df_valid["Imbalance_A01_MWh"].fillna(0)

        - df_valid["Imbalance_A02_MWh"].fillna(0)

    )


    df_valid["Physical_Imbalance_Abs_MWh"] = (

        df_valid["Physical_Imbalance_MWh"].abs()

    )


    df_valid["State"] = np.select(

        [

            df_valid["Physical_Imbalance_MWh"] > 0,

            df_valid["Physical_Imbalance_MWh"] < 0

        ],

        ["Deficit", "Surplus"],

        default="Echilibru"

    )


    return df, df_valid


def add_task54_magnitude_zones(df_valid):

    percentiles = {

        "p25": df_valid["Physical_Imbalance_Abs_MWh"].quantile(0.25),

        "p50": df_valid["Physical_Imbalance_Abs_MWh"].quantile(0.50),

        "p75": df_valid["Physical_Imbalance_Abs_MWh"].quantile(0.75),

        "p90": df_valid["Physical_Imbalance_Abs_MWh"].quantile(0.90),

        "p95": df_valid["Physical_Imbalance_Abs_MWh"].quantile(0.95),

        "p99": df_valid["Physical_Imbalance_Abs_MWh"].quantile(0.99),

    }


    labels = [

        "0–P25", "P25–P50", "P50–P75", "P75–P90",

        "P90–P95", "P95–P99", "P99+"

    ]


    bins = [

        -np.inf,

        percentiles["p25"],

        percentiles["p50"],

        percentiles["p75"],

        percentiles["p90"],

        percentiles["p95"],

        percentiles["p99"],

        np.inf

    ]


    df_valid["Imbalance_Zone"] = pd.cut(

        df_valid["Physical_Imbalance_Abs_MWh"],

        bins=bins,

        labels=labels,

        include_lowest=True

    )


    return df_valid, percentiles, labels


def build_task54_tables(df_valid, percentiles):

    percentile_table = pd.DataFrame({

        "Percentila": ["P25", "P50", "P75", "P90", "P95", "P99"],

        "Magnitudine_dezechilibru_MWh": [

            percentiles["p25"], percentiles["p50"], percentiles["p75"],

            percentiles["p90"], percentiles["p95"], percentiles["p99"]

        ]

    })


    state_table = (

        df_valid.groupby("State")

        .agg(

            N=("Imbalance_Price_EUR_MWh", "size"),

            Magnitudine_medie_MWh=("Physical_Imbalance_Abs_MWh", "mean"),

            Magnitudine_mediana_MWh=("Physical_Imbalance_Abs_MWh", "median"),

            Magnitudine_P95_MWh=("Physical_Imbalance_Abs_MWh", lambda x: x.quantile(0.95)),

            Pret_mediu_EUR_MWh=("Imbalance_Price_EUR_MWh", "mean"),

            Pret_median_EUR_MWh=("Imbalance_Price_EUR_MWh", "median"),

            Pret_std_EUR_MWh=("Imbalance_Price_EUR_MWh", "std"),

            Pret_min_EUR_MWh=("Imbalance_Price_EUR_MWh", "min"),

            Pret_max_EUR_MWh=("Imbalance_Price_EUR_MWh", "max")

        )

        .reset_index()

    )


    zone_table = (

        df_valid.groupby("Imbalance_Zone", observed=False)

        .agg(

            N=("Imbalance_Price_EUR_MWh", "size"),

            Magnitudine_medie_MWh=("Physical_Imbalance_Abs_MWh", "mean"),

            Pret_mediu_EUR_MWh=("Imbalance_Price_EUR_MWh", "mean"),

            Pret_median_EUR_MWh=("Imbalance_Price_EUR_MWh", "median"),

            Pret_std_EUR_MWh=("Imbalance_Price_EUR_MWh", "std"),

            Pret_P95_EUR_MWh=("Imbalance_Price_EUR_MWh", lambda x: x.quantile(0.95)),

            Pret_min_EUR_MWh=("Imbalance_Price_EUR_MWh", "min"),

            Pret_max_EUR_MWh=("Imbalance_Price_EUR_MWh", "max")

        )

        .reset_index()

    )


    direction_zone_table = (

        df_valid.groupby(["State", "Imbalance_Zone"], observed=False)

        .agg(

            N=("Imbalance_Price_EUR_MWh", "size"),

            Magnitudine_medie_MWh=("Physical_Imbalance_Abs_MWh", "mean"),

            Pret_median_EUR_MWh=("Imbalance_Price_EUR_MWh", "median"),

            Pret_mediu_EUR_MWh=("Imbalance_Price_EUR_MWh", "mean"),

            Pret_std_EUR_MWh=("Imbalance_Price_EUR_MWh", "std"),

            Pret_P95_EUR_MWh=("Imbalance_Price_EUR_MWh", lambda x: x.quantile(0.95))

        )

        .reset_index()

    )


    return percentile_table, state_table, zone_table, direction_zone_table


def calculate_task54_correlations(df_valid):

    pearson_signed = df_valid["Physical_Imbalance_MWh"].corr(

        df_valid["Imbalance_Price_EUR_MWh"], method="pearson"

    )

    spearman_signed = df_valid["Physical_Imbalance_MWh"].corr(

        df_valid["Imbalance_Price_EUR_MWh"], method="spearman"

    )

    pearson_abs = df_valid["Physical_Imbalance_Abs_MWh"].corr(

        df_valid["Imbalance_Price_EUR_MWh"], method="pearson"

    )

    spearman_abs = df_valid["Physical_Imbalance_Abs_MWh"].corr(

        df_valid["Imbalance_Price_EUR_MWh"], method="spearman"

    )


    correlation_table = pd.DataFrame({

        "Variabila": ["Dezechilibru semnat", "Magnitudinea dezechilibrului"],

        "Pearson": [pearson_signed, pearson_abs],

        "Spearman": [spearman_signed, spearman_abs]

    })


    return correlation_table, pearson_signed, spearman_signed, pearson_abs, spearman_abs


def calculate_task54_regression(df_valid):

    regression_rows = []


    for state in ["Deficit", "Surplus"]:

        d = df_valid[df_valid["State"] == state].copy()

        x = d["Physical_Imbalance_Abs_MWh"].to_numpy()

        y = d["Imbalance_Price_EUR_MWh"].to_numpy()


        mask = np.isfinite(x) & np.isfinite(y)

        x = x[mask]

        y = y[mask]


        coef = np.polyfit(x, y, 2)

        y_pred = np.polyval(coef, x)

        ss_res = np.sum((y - y_pred) ** 2)

        ss_tot = np.sum((y - y.mean()) ** 2)

        r2 = 1 - ss_res / ss_tot


        regression_rows.append({

            "State": state,

            "a_x2": coef[0],

            "b_x": coef[1],

            "c": coef[2],

            "R2": r2,

            "Ecuație": (

                f"Price = {coef[0]:.4f} × Imbalance² "

                f"+ {coef[1]:.4f} × Imbalance + {coef[2]:.4f}"

            )

        })


    return pd.DataFrame(regression_rows)


def calculate_task54_zone_changes(zone_table):

    zone_change = zone_table.copy()

    zone_change["Schimbare_pret_median"] = zone_change["Pret_median_EUR_MWh"].diff()

    zone_change["Schimbare_pret_std"] = zone_change["Pret_std_EUR_MWh"].diff()

    zone_change["Schimbare_magnitudine"] = zone_change["Magnitudine_medie_MWh"].diff()

    zone_change["Pret_median_per_MWh_dezechilibru"] = (

        zone_change["Pret_median_EUR_MWh"] /

        zone_change["Magnitudine_medie_MWh"]

    )

    return zone_change


def print_task54_outputs(df, df_valid, percentile_table, state_table, zone_table,

                         direction_zone_table, correlation_table, regression_table,

                         zone_change, labels, pearson_signed, spearman_signed,

                         pearson_abs):

    print("DATE ANALIZATE")

    print("-" * 80)

    print(f"Observații totale în fișier: {len(df):,}")

    print(f"Observații analizate: {len(df_valid):,}")

    print(f"Deficit: {(df_valid['State'] == 'Deficit').sum():,}")

    print(f"Surplus: {(df_valid['State'] == 'Surplus').sum():,}")

    print(f"Echilibru: {(df_valid['State'] == 'Echilibru').sum():,}")

    print()


    print("1. PERCENTILELE MAGNITUDINII DEZECHILIBRULUI")

    print("-" * 80)

    display(percentile_table.round(3))


    print("2. COMPARAȚIE DEFICIT vs SURPLUS")

    print("-" * 80)

    display(state_table.round(3))


    print("3. PREȚUL ÎN FUNCȚIE DE MAGNITUDINEA DEZECHILIBRULUI")

    print("-" * 80)

    display(zone_table.round(3))


    print("4. DEFICIT / SURPLUS PE PERCENTILE")

    print("-" * 80)

    display(direction_zone_table.round(3))


    print("5. RELAȚIA DEZECHILIBRU – PREȚ")

    print("-" * 80)

    display(correlation_table.round(4))


    print("6. REGRESIE NELINIARĂ – GRAD 2")

    print("-" * 80)

    display(regression_table.round(4))


    print("7. IDENTIFICAREA ZONELOR CU VARIAȚII DISPROPORȚIONATE")

    print("-" * 80)

    display(zone_change[

        [

            "Imbalance_Zone", "N", "Magnitudine_medie_MWh",

            "Pret_median_EUR_MWh", "Schimbare_pret_median",

            "Pret_std_EUR_MWh", "Schimbare_pret_std"

        ]

    ].round(3))


    p99_zone = zone_table[zone_table["Imbalance_Zone"] == "P99+"]

    p25_zone = zone_table[zone_table["Imbalance_Zone"] == "0–P25"]


    print("8. ZONA EXTREMĂ P99+")

    print("-" * 80)


    if not p99_zone.empty and not p25_zone.empty:

        median_p25 = float(p25_zone["Pret_median_EUR_MWh"].iloc[0])

        median_p99 = float(p99_zone["Pret_median_EUR_MWh"].iloc[0])

        increase = median_p99 - median_p25

        print(f"Preț median 0–P25 : {median_p25:.2f} EUR/MWh")

        print(f"Preț median P99+  : {median_p99:.2f} EUR/MWh")

        print(f"Variația medianei  : {increase:.2f} EUR/MWh")


def plot_task54_graphs(df_valid, direction_zone_table, graph1, graph2):
    zones = [
        "0–P25", "P25–P50", "P50–P75", "P75–P90",
        "P90–P95", "P95–P99", "P99+"
    ]

    fig1 = go.Figure()

    for state in ["Deficit", "Surplus"]:
        d = direction_zone_table[
            direction_zone_table["State"] == state
        ].copy()

        d["Imbalance_Zone"] = pd.Categorical(
            d["Imbalance_Zone"],
            categories=zones,
            ordered=True
        )
        d = d.sort_values("Imbalance_Zone")

        fig1.add_trace(
            go.Scatter(
                x=d["Imbalance_Zone"].astype(str),
                y=d["Pret_median_EUR_MWh"],
                mode="lines+markers",
                name=state
            )
        )

    fig1.update_layout(
        title="Task 54 – Preț median în funcție de direcția și magnitudinea dezechilibrului",
        xaxis_title="Magnitudinea dezechilibrului – percentile",
        yaxis_title="Preț median [EUR/MWh]",
        template="plotly_white",
        width=1100,
        height=600
    )
    fig1.update_yaxes(showgrid=True)
    fig1.show()

    fig2 = go.Figure()

    for state in ["Deficit", "Surplus"]:
        d = df_valid[df_valid["State"] == state]

        fig2.add_trace(
            go.Scatter(
                x=d["Physical_Imbalance_MWh"],
                y=d["Imbalance_Price_EUR_MWh"],
                mode="markers",
                marker=dict(size=5, opacity=0.20),
                name=state
            )
        )

    fig2.add_vline(
        x=0,
        line_dash="dash",
        line_width=1
    )
    fig2.update_layout(
        title="Task 54 – Relația dintre dezechilibrul fizic și prețul de dezechilibru",
        xaxis_title="Dezechilibru fizic semnat A01 − A02 [MWh]",
        yaxis_title="Preț de dezechilibru [EUR/MWh]",
        template="plotly_white",
        width=1100,
        height=600
    )
    fig2.update_yaxes(showgrid=True)
    fig2.show()

def export_task54(df_valid, percentile_table, state_table, zone_table,

                  direction_zone_table, correlation_table, regression_table,

                  zone_change, paths):

    df_valid.to_csv(paths["input_csv"], index=False)

    zone_table.round(6).to_csv(paths["analysis_csv"], index=False)


    df_valid_excel = df_valid.copy()

    df_valid_excel["Timestamp"] = (

        pd.to_datetime(df_valid_excel["Timestamp"], utc=True)

        .dt.tz_localize(None)

    )

    df_valid_excel.to_excel(paths["raw_xlsx"], index=False, engine="openpyxl")


    with pd.ExcelWriter(paths["analysis_xlsx"], engine="openpyxl") as writer:

        percentile_table.round(6).to_excel(writer, sheet_name="Percentile", index=False)

        state_table.round(6).to_excel(writer, sheet_name="Deficit_Surplus", index=False)

        zone_table.round(6).to_excel(writer, sheet_name="Magnitude_Zones", index=False)

        direction_zone_table.round(6).to_excel(writer, sheet_name="Direction_Zones", index=False)

        correlation_table.round(6).to_excel(writer, sheet_name="Correlations", index=False)

        regression_table.round(6).to_excel(writer, sheet_name="Nonlinear_Regression", index=False)

        zone_change.round(6).to_excel(writer, sheet_name="Disproportionate_Zones", index=False)


def run_task54():

    paths = configure_task54()

    df, df_valid = load_task54_data(paths["input_csv"])

    df_valid, percentiles, labels = add_task54_magnitude_zones(df_valid)


    (

        percentile_table,

        state_table,

        zone_table,

        direction_zone_table,

    ) = build_task54_tables(df_valid, percentiles)


    (

        correlation_table,

        pearson_signed,

        spearman_signed,

        pearson_abs,

        spearman_abs,

    ) = calculate_task54_correlations(df_valid)


    regression_table = calculate_task54_regression(df_valid)

    zone_change = calculate_task54_zone_changes(zone_table)


    print_task54_outputs(

        df, df_valid, percentile_table, state_table, zone_table,

        direction_zone_table, correlation_table, regression_table,

        zone_change, labels, pearson_signed, spearman_signed, pearson_abs

    )


    plot_task54_graphs(

        df_valid, direction_zone_table,

        paths["graph1"], paths["graph2"]

    )


    export_task54(

        df_valid, percentile_table, state_table, zone_table,

        direction_zone_table, correlation_table, regression_table,

        zone_change, paths

    )


    print()

    print("=" * 80)

    print("CONCLUZIA TASK 54")

    print("=" * 80)

    print(f"• Pearson dezechilibru semnat – preț: {pearson_signed:.4f}")

    print(f"• Spearman dezechilibru semnat – preț: {spearman_signed:.4f}")

    print(f"• Pearson magnitudine – preț: {pearson_abs:.4f}")

    print("• Direcția dezechilibrului are o relație mult mai puternică cu prețul decât magnitudinea singură.")

    print("• Analiza pe percentile identifică zonele de dezechilibru extrem.")

    print("• Regresia polinomială de grad 2 testează relația neliniară pentru Deficit și Surplus.")

    print("• Graficele evidențiază relația dintre direcție, magnitudine și preț.")

    print()

    print("=" * 80)

    print("FIȘIERE SALVATE")

    print("=" * 80)

    print(paths["raw_xlsx"])

    print(paths["analysis_csv"])

    print(paths["analysis_xlsx"])

    print(paths["graph1"])

    print(paths["graph2"])

    print()

    print("✅ TASK 54 FINALIZAT FĂRĂ CALL API ENTSO-E")

    print("=" * 80)


run_task54()


### Concluzii

- Dezechilibrul semnat prezintă o corelație inversă masivă cu prețul de dezechilibru, în timp ce magnitudinea brută are o corelație nulă.

- Excedentul de energie urcă prețul median.

- Lipsa de energie împinge prețul median în negativ.
